In [ ]:
# @title
!pip install bert-score
!pip install datasets
!pip install sentence-transformers datasets accelerate transformers wandb
!pip install sentence-transformers torch sklearn bert-score
!pip install sentencepiece
!pip install -q sentencepiece transformers sentence-transformers bert-score torch scikit-learn pandas numpy
!pip install -q pytorch-crf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


Preprocessing (FEMSeg_kaz)


In [ ]:
# @title
# ============================================================
# paraphrase-multilingual-MiniLM-L12-v2 + FEMSeg_v3
# BASE_ONLY НЕГІЗІНДЕ ҚАЙТА ҚҰРЫЛЫМДАЛҒАН (АДАЛ САЛЫСТЫРУ)
# MLM жоқ, held-out TEST, FINE-TUNE ЖОҚ, dual-view (эмбеддинг деңгейінде)
#
# Metrics (baseline-мен бірдей шарттар):
#   Exact@1
#   TokenF1@1
#   MeanCos@1(QSim)
#   Semantic@1(ans_cos>=0.85)
#   BERTScoreF1@1
# ============================================================

# =========================
# ENV
# =========================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_DISABLED"] = "true"

import re
import csv
import glob
import time
import math
import json
import random
import string
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from bert_score import score as bert_score
except Exception:
    bert_score = None

try:
    from torchcrf import CRF
except Exception:
    CRF = None

from sentence_transformers import SentenceTransformer, models
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# --------------------------
# Детерминизм
# --------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass
try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥 DEVICE:", DEVICE)

# --------------------------
# CONFIG
# --------------------------
DATA_JSON_PATH = "50k.json"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

TEST_SIZE = 0.10
SEM_THR = 0.85

ENCODE_BATCH_SIZE = 64
SIM_BATCH_SIZE = 256
PRINT_PROGRESS_EVERY = 5000

EXPORT_CSV = True
CSV_PATH = "femseg_aligned_test_details.csv"

# --------------------------
# Dual-view конфигурациясы
# --------------------------
ALPHA_SEG = 0.4

# ============================================================
# 0) QA деректері – JSON файлдан robust оқу
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p
    candidates = [f"/content/{p}", f"/content/drive/MyDrive/{p}"]
    for c in candidates:
        if Path(c).exists():
            return c
    name = Path(p).name
    hits = glob.glob(f"**/{name}", recursive=True)
    if hits:
        return hits[0]
    near = glob.glob("**/*.json", recursive=True)
    raise FileNotFoundError(
        f"❌ File not found: {p}\nPWD: {Path.cwd()}\n"
        f"Found .json (first 30):\n" + "\n".join(near[:30])
    )


def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)


def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)


def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text


def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None
    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    return None


def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)
    if normalized is not None:
        return [normalized]
    if isinstance(obj, list):
        out = []
        for item in obj:
            norm = _normalize_record(item)
            if norm is not None:
                out.append(norm)
        return out
    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                out = []
                for item in obj[key]:
                    norm = _normalize_record(item)
                    if norm is not None:
                        out.append(norm)
                return out
    return []


def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    obj = json.loads(_normalize_json_text(text))
    records = _extract_records(obj)
    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")
    return records


def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx = 0
    records = []
    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1
        if idx >= len(text):
            break
        obj, end = decoder.raw_decode(text, idx)
        idx = end
        extracted = _extract_records(obj)
        if extracted:
            records.extend(extracted)
    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")
    return records


def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records = []
    bad_lines = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()
        if not s:
            continue
        if s.endswith(","):
            s = s[:-1].strip()
        s = _normalize_json_text(s)
        try:
            obj = json.loads(s)
            extracted = _extract_records(obj)
            if extracted:
                records.extend(extracted)
            else:
                bad_lines.append((line_no, "QA өрістері табылмады", s[:200]))
        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))
    if not records:
        preview = "\n".join(
            [f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]]
        )
        raise ValueError(
            "Файлдан бірде-бір QA жазба оқылмады.\n"
            f"Алғашқы қате жолдар:\n{preview}"
        )
    if bad_lines:
        print(f"⚠ Ескерту: {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")
    return records


def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []
    for parser in (
        _parse_as_single_json,
        _parse_as_multiple_json_objects,
        _parse_line_by_line,
    ):
        try:
            records = parser(text)
            if records:
                return records
        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")
    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


data_path = find_data_path(DATA_JSON_PATH)
print(f"📥 QA JSON файлдан жүктеу: {data_path}")

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON-да question/answer немесе instruction/response болуы тиіс."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"]   = qa_data["answer"].astype(str).str.strip()
qa_data = qa_data[(qa_data["question"] != "") & (qa_data["answer"] != "")].reset_index(drop=True)

print(f"📚 Барлық QA жазба саны: {len(qa_data)}")
print(qa_data.head(3))

# ============================================================
# 1) Text normalization — BASELINE-МЕН БІРДЕЙ (clean_view)
# ============================================================

_punct_space_left  = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space       = re.compile(r"\s+")


def clean_view(text: str) -> str:
    """Baseline-мен бірдей нормализация."""
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def norm_for_exact(text: str) -> str:
    """Baseline-мен бірдей exact match нормализация."""
    return re.sub(r"\s+", " ", clean_view(text).lower()).strip()


def tokens(text: str) -> List[str]:
    """Baseline-мен бірдей токенизация."""
    t = clean_view(text).lower()
    return re.findall(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+", t)


def token_f1(pred: str, gold: str) -> float:
    """Baseline-мен бірдей TokenF1 есептеу."""
    p = tokens(pred)
    g = tokens(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc = Counter(p)
    gc = Counter(g)
    inter = sum((pc & gc).values())
    if inter == 0:
        return 0.0
    prec = inter / max(1, len(p))
    rec  = inter / max(1, len(g))
    return (2 * prec * rec) / (prec + rec + 1e-12)

# ============================================================
# 2) FEMSeg_v3 (KazMorphSegmentor)
# ============================================================

BMES_TAGS = ["B", "M", "E", "S"]
TAG2ID = {t: i for i, t in enumerate(BMES_TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}


class FEMSegV3(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int = 128,
        lstm_hidden_dim: int = 256,
        dropout: float = 0.3,
        num_tags: int = len(BMES_TAGS),
    ):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=char_emb_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward_features(self, x, mask):
        emb = self.emb(x)
        emb = self.dropout(emb)
        h, _ = self.lstm(emb)
        h = self.dropout(h)
        return h

    def forward_logits(self, x, mask):
        feats = self.forward_features(x, mask)
        return self.fc(feats)

    def forward(self, x, tags=None, mask=None):
        if mask is None:
            mask = x != 0
        emissions = self.forward_logits(x, mask)
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)


def bmes_to_cse(chars: List[str], tags: List[str]) -> str:
    morphs: List[str] = []
    cur = ""
    for ch, t in zip(chars, tags):
        if t == "B":
            if cur:
                morphs.append(cur)
            cur = ch
        elif t == "M":
            cur += ch
        elif t == "E":
            cur += ch
            morphs.append(cur)
            cur = ""
        elif t == "S":
            if cur:
                morphs.append(cur)
            morphs.append(ch)
            cur = ""
    if cur:
        morphs.append(cur)
    return "@@ ".join(morphs)


class KazMorphSegmentor:
    def __init__(self, char2id_path: str, ckpt_path: str, use_cuda: bool = True):
        with open(char2id_path, "r", encoding="utf-8") as f:
            self.char2id: Dict[str, int] = json.load(f)
        self.pad_id = 0
        self.unk_id = self.char2id.get("<unk>") or self.char2id.get("[UNK]") or 1
        self.device = torch.device(
            "cuda" if (use_cuda and torch.cuda.is_available()) else "cpu"
        )
        vocab_size = max(self.char2id.values()) + 1
        self.model = FEMSegV3(vocab_size=vocab_size)
        self.model.to(self.device)
        state = torch.load(ckpt_path, map_location=self.device)
        if isinstance(state, dict):
            if "state_dict" in state:
                state = state["state_dict"]
            elif "model_state_dict" in state:
                state = state["model_state_dict"]
        if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        self.word_cache: Dict[str, str] = {}
        self.text_cache: Dict[str, str] = {}

    def _word_to_tensor(self, word: str):
        ids = [self.char2id.get(ch, self.unk_id) for ch in list(word)]
        x = torch.tensor([ids], dtype=torch.long, device=self.device)
        mask = (x != self.pad_id)
        return x, mask

    def segment_word(self, word: str) -> str:
        word = word.strip()
        if not word:
            return ""
        cached = self.word_cache.get(word)
        if cached is not None:
            return cached
        x, mask = self._word_to_tensor(word)
        with torch.no_grad():
            best_paths = self.model(x, mask=mask)
        tags = [ID2TAG[i] for i in best_paths[0]]
        result = bmes_to_cse(list(word), tags)
        self.word_cache[word] = result
        return result

    def segment_text(self, text: str) -> str:
        text = text.strip()
        if not text:
            return ""
        cached = self.text_cache.get(text)
        if cached is not None:
            return cached
        words = text.split()
        out_words = [self.segment_word(w) for w in words]
        result = " | ".join(out_words)
        self.text_cache[text] = result
        return result


def femseg_to_sp(text: str) -> str:
    """
    FEMSeg нәтижесін SentencePiece-тәрізді формата айналдыру.
    clean_view арқылы алдын ала тазаланады (baseline-мен үйлесімді).
    """
    text = clean_view(str(text))
    if not text:
        return ""
    seg_line = femseg.segment_text(text)
    sp_tokens = []
    for word_seg in seg_line.split(" | "):
        word_seg = word_seg.strip()
        if not word_seg:
            continue
        morphs = [m.strip() for m in word_seg.split("@@ ") if m.strip()]
        if not morphs:
            continue
        sp_tokens.append("▁" + morphs[0])
        sp_tokens.extend(morphs[1:])
    return " ".join(sp_tokens)


# ============================================================
# FEMSeg модель жолдары
# ============================================================
if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)

FEMSEG_CHAR2ID = "/content/drive/MyDrive/KAZ_MORPH/char2id_femseg_v3_50k.json"
FEMSEG_CKPT    = "/content/drive/MyDrive/KAZ_MORPH/femseg_v3_50k_epoch3.pt"

assert os.path.exists(FEMSEG_CHAR2ID), f"char2id_femseg_v3_50k.json табылмады: {FEMSEG_CHAR2ID}"
assert os.path.exists(FEMSEG_CKPT),    f"femseg_v3_50k_epoch3.pt табылмады: {FEMSEG_CKPT}"

print("⬇️ FEMSeg_v3 сегментаторын жүктеу...")
femseg = KazMorphSegmentor(
    char2id_path=FEMSEG_CHAR2ID,
    ckpt_path=FEMSEG_CKPT,
    use_cuda=True,
)
print("✅ FEMSeg_v3 дайын.")
print("🔎 FEMSeg тест:", femseg_to_sp("Мен мектепке барамын"))

# ============================================================
# 3) MiniLM SentenceTransformer
# ============================================================
print("⬇️ MiniLM encoder-ін жүктеу:", MODEL_NAME)
retr_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
retr_model.eval()
print("✅ MiniLM моделі дайын.")

# ============================================================
# CLEAN view + SEG-ді БІР РЕТ қана дайындау
# ============================================================

qa_data["q_clean"] = qa_data["question"].map(clean_view)
qa_data["a_clean"] = qa_data["answer"].map(clean_view)

print(f"\n🧩 FEMSeg сегментация басталды... ({len(qa_data)} жазба)")
t0 = time.time()

unique_questions = qa_data["q_clean"].unique().tolist()
q_sp_map = {}
for i, txt in enumerate(unique_questions, start=1):
    q_sp_map[txt] = femseg_to_sp(txt)
    if i % PRINT_PROGRESS_EVERY == 0 or i == len(unique_questions):
        print(f"   {i}/{len(unique_questions)} дайын | {time.time() - t0:.1f} сек")

qa_data["q_sp"] = qa_data["q_clean"].map(q_sp_map)

print("\n🔎 Мысал:")
print(qa_data[["question", "q_clean", "q_sp", "answer", "a_clean"]].head())

# ============================================================
# TRAIN / TEST SPLIT — baseline-мен бірдей параметрлер
# ============================================================
train_df, test_df = train_test_split(
    qa_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_idx = train_df.index.to_numpy()
test_idx  = test_df.index.to_numpy()

print(f"\n📊 TRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED} | test_size={TEST_SIZE}")

train_questions = train_df["question"].tolist()
train_answers   = train_df["a_clean"].tolist()   # baseline-мен бірдей: a_clean
test_questions  = test_df["question"].tolist()
test_answers    = test_df["a_clean"].tolist()     # baseline-мен бірдей: a_clean

# ============================================================
# Энкодер көмекші функциялары
# ============================================================

def _encode_dual(clean_texts: List[str], seg_texts: List[str],
                 batch_size: int = ENCODE_BATCH_SIZE,
                 normalize: bool = True) -> np.ndarray:
    """
    Dual-view: ALPHA_SEG * seg_emb + (1 - ALPHA_SEG) * clean_emb
    Normalize=True болса — baseline-мен бірдей нормаланған вектор.
    """
    clean_texts = [str(t) for t in clean_texts]
    seg_texts   = [str(t) for t in seg_texts]

    v_seg = retr_model.encode(
        seg_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    v_clean = retr_model.encode(
        clean_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    vecs = ALPHA_SEG * v_seg + (1.0 - ALPHA_SEG) * v_clean

    if normalize:
        norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
        vecs = vecs / norms

    return vecs.astype(np.float32)


def _encode_retr(texts: List[str],
                 batch_size: int = ENCODE_BATCH_SIZE,
                 normalize: bool = True) -> np.ndarray:
    """Runtime кодтау: clean_view + femseg → dual-view."""
    clean = [clean_view(t) for t in texts]
    seg   = [femseg_to_sp(t) for t in clean]
    return _encode_dual(clean, seg, batch_size=batch_size, normalize=normalize)


def _encode_eval_answers(texts: List[str],
                         batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
    """
    Answer embedding үшін — baseline-мен бірдей:
    clean_view + normalize=True.
    """
    clean = [clean_view(t) for t in texts]
    vecs = retr_model.encode(
        clean,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,     # baseline: normalize=True
        show_progress_bar=False,
    )
    return vecs.astype(np.float32)

# ============================================================
# FULL question embedding-терді БІР РЕТ қана есептеу
# ============================================================

qa_questions_full = qa_data["q_clean"].tolist()
qa_q_sp_full      = qa_data["q_sp"].tolist()
qa_answers_full   = qa_data["a_clean"].tolist()

print("\n📦 FULL QA үшін dual-view embedding-тер есептелуде...")
t0 = time.time()
qa_q_emb_full = _encode_dual(
    clean_texts=qa_questions_full,
    seg_texts=qa_q_sp_full,
    batch_size=ENCODE_BATCH_SIZE,
    normalize=True,
)
print(f"✅ FULL QA dual-view embedding дайын: {time.time() - t0:.2f} сек")

train_q_emb = qa_q_emb_full[train_idx]
test_q_emb  = qa_q_emb_full[test_idx]

# ============================================================
# Retrieval функциялары
# ============================================================

_query_emb_cache: Dict[str, np.ndarray] = {}


def _get_query_embedding(question: str) -> np.ndarray:
    question = clean_view(question)
    cached = _query_emb_cache.get(question)
    if cached is not None:
        return cached
    q_sp = femseg_to_sp(question)
    qv = _encode_dual([question], [q_sp], batch_size=1, normalize=True)
    _query_emb_cache[question] = qv
    return qv


def _retrieve_from_kb(
    question: str,
    kb_q_emb: np.ndarray,
    kb_answers: List[str],
    threshold: float = 0.0,
):
    qv = _get_query_embedding(question)
    sims = (kb_q_emb @ qv.T).squeeze(1)
    idx = int(np.argmax(sims))
    sim_val = float(sims[idx])

    if sim_val < threshold:
        return "Кешіріңіз, нақты жауап табылмады.", -1, sim_val

    return kb_answers[idx], idx, sim_val


def ask_question_runtime(question: str, threshold: float = 0.0):
    """Runtime режимі: FULL KB бойынша."""
    return _retrieve_from_kb(
        question,
        kb_q_emb=qa_q_emb_full,
        kb_answers=qa_answers_full,
        threshold=threshold,
    )


def ask_question_eval(question: str):
    """Eval режимі: тек TRAIN KB бойынша."""
    ans, idx, sim = _retrieve_from_kb(
        question,
        kb_q_emb=train_q_emb,
        kb_answers=train_answers,
        threshold=0.0,
    )
    return ans, idx, sim

# ============================================================
# BERTScore helper — baseline-мен бірдей
# ============================================================

def _bert_lang_try(preds: List[str], golds: List[str]) -> Optional[float]:
    """
    Baseline-мен бірдей: lang параметрімен fallback (kk → tr → en),
    rescale_with_baseline=True.
    """
    if bert_score is None:
        return None
    for lang in ("kk", "tr", "en"):
        try:
            _, _, F1 = bert_score(preds, golds, lang=lang, rescale_with_baseline=True)
            arr = F1.numpy() if hasattr(F1, "numpy") else np.array(F1)
            return float(np.mean(arr))
        except Exception:
            continue
    return None

# ============================================================
# TEST prediction-дерін БІР РЕТ қана есептеу
# ============================================================

_eval_cache: Dict[str, Any] = {}


def precompute_eval_predictions():
    if "pred_answers" in _eval_cache:
        return

    print("\n⚡ TEST → TRAIN retrieval бір рет есептелуде...")
    t0 = time.time()

    pred_indices = []
    pred_sims = []

    n_test = len(test_q_emb)

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]
        sims = batch @ train_q_emb.T
        idxs = np.argmax(sims, axis=1)
        vals = sims[np.arange(end - start), idxs]

        pred_indices.extend(idxs.tolist())
        pred_sims.extend(vals.tolist())

    pred_answers = [train_answers[i] for i in pred_indices]

    _eval_cache["pred_indices"]  = np.array(pred_indices, dtype=np.int32)
    _eval_cache["pred_q_sims"]   = np.array(pred_sims, dtype=np.float32)
    _eval_cache["pred_answers"]  = pred_answers
    _eval_cache["true_answers"]  = test_answers
    _eval_cache["test_questions"] = [clean_view(q) for q in test_questions]

    print(f"✅ TEST prediction дайын: {time.time() - t0:.2f} сек")


def _encode_answer_pairs_for_eval():
    if "pred_eval_emb" in _eval_cache and "true_eval_emb" in _eval_cache:
        return _eval_cache["pred_eval_emb"], _eval_cache["true_eval_emb"]

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n📐 Answer embedding-тері есептелуде...")
    t0 = time.time()

    # Baseline-мен бірдей: _encode_eval_answers (normalize=True)
    pv = _encode_eval_answers(preds)
    tv = _encode_eval_answers(trues)

    _eval_cache["pred_eval_emb"] = pv
    _eval_cache["true_eval_emb"] = tv

    print(f"✅ Answer embedding дайын: {time.time() - t0:.2f} сек")
    return pv, tv

# ============================================================
# Метрикалар — baseline-мен бірдей логика
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()
    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    scores = [float(norm_for_exact(p) == norm_for_exact(t))
              for p, t in zip(preds, trues)]
    return float(np.mean(scores))


def evaluate_token_f1_at_1():
    precompute_eval_predictions()
    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    scores = [token_f1(p, t) for p, t in zip(preds, trues)]
    return float(np.mean(scores))


def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()
    return float(np.mean(_eval_cache["pred_q_sims"]))


def evaluate_semantic_at_1_ans_cos(threshold: float = SEM_THR):
    pv, tv = _encode_answer_pairs_for_eval()
    # Baseline-мен бірдей: normalize=True болғандықтан dot = cosine
    ans_cos = np.sum(pv * tv, axis=1)
    _eval_cache["ans_cos"] = ans_cos
    return float(np.mean(ans_cos >= threshold))


def evaluate_bertscore_f1_at_1():
    if "bertscore_f1_mean" in _eval_cache:
        return float(_eval_cache["bertscore_f1_mean"])

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n📊 BERTScoreF1@1 есептелуде...")
    t0 = time.time()

    # Baseline-мен бірдей: lang fallback, rescale_with_baseline=True
    f1_mean = _bert_lang_try(preds, trues)
    _eval_cache["bertscore_f1_mean"] = f1_mean

    print(f"✅ BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")
    return f1_mean


def build_eval_details():
    if "details_df" in _eval_cache:
        return _eval_cache["details_df"]

    precompute_eval_predictions()

    preds   = _eval_cache["pred_answers"]
    trues   = _eval_cache["true_answers"]
    qsims   = _eval_cache["pred_q_sims"]

    if "ans_cos" not in _eval_cache:
        _ = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)

    ans_cos = _eval_cache["ans_cos"]
    rows = []

    for q, gold, pred, qsim, acos in zip(
        _eval_cache["test_questions"], trues, preds, qsims, ans_cos
    ):
        ex = float(norm_for_exact(pred) == norm_for_exact(gold))
        f1 = float(token_f1(pred, gold))
        sh = float(acos >= SEM_THR)
        rows.append({
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,
            "QSim": float(qsim),
            "Exact": ex,
            "TokenF1": f1,
            "AnsCos": float(acos),
            "SemHit": sh,
        })

    details_df = pd.DataFrame(rows)
    _eval_cache["details_df"] = details_df
    return details_df


def export_details_csv(path: str):
    details_df = build_eval_details()
    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV exported: {path}  (rows={len(details_df)})")


def evaluate_system_femseg():
    t0 = time.time()

    precompute_eval_predictions()

    exact_at_1    = evaluate_exact_at_1()
    token_f1_at_1 = evaluate_token_f1_at_1()
    mean_cos_qsim = evaluate_mean_cos_qsim_at_1()
    semantic_at_1 = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    bertscore_f1  = evaluate_bertscore_f1_at_1()

    total_t = time.time() - t0

    print("\n==================== RESULTS (FEMSeg+MiniLM / HELD-OUT TEST) ====================")
    print(f"                     Dataset: FEMSeg_DUAL_VIEW")
    print(f"                     Exact@1: {exact_at_1:.6f}")
    print(f"                   TokenF1@1: {token_f1_at_1:.6f}")
    print(f"             MeanCos@1(QSim): {mean_cos_qsim:.6f}")
    print(f"    Semantic@1(ans_cos≥{SEM_THR}): {semantic_at_1:.6f}")

    if bertscore_f1 is not None:
        print(f"               BERTScoreF1@1: {bertscore_f1:.6f}")
    else:
        print(f"               BERTScoreF1@1: N/A (bert-score орнатылмаған)")

    print(f"               EvaluationTime: {total_t:.2f} sec")
    print(f"                    ALPHA_SEG: {ALPHA_SEG:.2f}")

    if EXPORT_CSV:
        export_details_csv(CSV_PATH)

# ============================================================
# Негізгі блок
# ============================================================

if __name__ == "__main__":
    try:
        while True:
            user_input = input("\nСұрақ енгізіңіз (шығу үшін 'exit'): ")
            if user_input.strip().lower() == "exit":
                print("Бағдарлама тоқтатылды. 👋")
                break

            answer, idx, sim = ask_question_runtime(user_input, threshold=0.0)
            print("\n=== [FEMSeg+MiniLM ЖАУАП] ===")
            print(answer)
            print(f"(similarity={sim:.4f}, idx={idx})")

    except EOFError:
        pass

    evaluate_system_femseg()

🖥 DEVICE: cuda
📥 QA JSON файлдан жүктеу: 50k.json
⚠ Ескерту: 256 жол оқылмады, бірақ қалғандары жүктелді.
📚 Барлық QA жазба саны: 50275
                                  question  \
0                     Python дегеніміз не?   
1           Python тілі қалай пайда болды?   
2  Python тілінің басты ерекшелігі қандай?   

                                              answer  
0  Python — қарапайым және оқуға оңай бағдарламал...  
1  Python тілін 1991 жылы Гвидо ван Россум жасаға...  
2  Python — оқуға жеңіл және интуитивті синтаксис...  
Mounted at /content/drive
⬇️ FEMSeg_v3 сегментаторын жүктеу...
✅ FEMSeg_v3 дайын.
🔎 FEMSeg тест: ▁Мен ▁мектеп ке ▁бар а мын
⬇️ MiniLM encoder-ін жүктеу: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ MiniLM моделі дайын.

🧩 FEMSeg сегментация басталды... (50275 жазба)
   5000/48908 дайын | 12.1 сек
   10000/48908 дайын | 18.5 сек
   15000/48908 дайын | 22.6 сек
   20000/48908 дайын | 25.6 сек
   25000/48908 дайын | 27.9 сек
   30000/48908 дайын | 32.6 сек
   35000/48908 дайын | 34.8 сек
   40000/48908 дайын | 38.3 сек
   45000/48908 дайын | 53.7 сек
   48908/48908 дайын | 65.3 сек

🔎 Мысал:
                                   question  \
0                      Python дегеніміз не?   
1            Python тілі қалай пайда болды?   
2   Python тілінің басты ерекшелігі қандай?   
3  Python-да айнымалылар қалай жарияланады?   
4       Python интерпретаторы дегеніміз не?   

                                    q_clean  \
0                      Python дегеніміз не?   
1            Python тілі қалай пайда болды?   
2   Python тілінің басты ерекшелігі қандай?   
3  Python-да айнымалылар қалай жарияланады?   
4       Python интерпретаторы дегеніміз не?   

                                  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScoreF1@1 дайын: 58.57 сек

==================== RESULTS (FEMSeg+MiniLM / HELD-OUT TEST) ====================
                     Dataset: FEMSeg_DUAL_VIEW
                     Exact@1: 0.014519
                   TokenF1@1: 0.452828
             MeanCos@1(QSim): 0.897212
    Semantic@1(ans_cos≥0.85): 0.340692
               BERTScoreF1@1: 0.828705
               EvaluationTime: 76.62 sec
                    ALPHA_SEG: 0.40

✅ CSV exported: femseg_aligned_test_details.csv  (rows=5028)
